In [1]:
import sys
import mediapipe as mp

print("Executing from:", sys.executable)
print("Loading MediaPipe from:", mp.__file__)

Executing from: c:\Users\adity\anaconda3\envs\Myfirst\python.exe
Loading MediaPipe from: c:\Users\adity\anaconda3\envs\Myfirst\lib\site-packages\mediapipe\__init__.py


In [1]:
import cv2
import mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python import vision

# ── download model if needed ───────────────────────────────────────────────────
import urllib.request, os
if not os.path.exists("hand_landmarker.task"):
    print("Downloading model...")
    urllib.request.urlretrieve(
        "https://storage.googleapis.com/mediapipe-models/hand_landmarker/hand_landmarker/float16/1/hand_landmarker.task",
        "hand_landmarker.task"
    )
    print("Done.")

# ── hand connections (which landmarks to draw lines between) ───────────────────
CONNECTIONS = [
    (0,1),(1,2),(2,3),(3,4),         # thumb
    (0,5),(5,6),(6,7),(7,8),         # index
    (0,9),(9,10),(10,11),(11,12),    # middle
    (0,13),(13,14),(14,15),(15,16),  # ring
    (0,17),(17,18),(18,19),(19,20),  # pinky
    (5,9),(9,13),(13,17)             # palm
]

def draw_hands(frame, result):
    h, w = frame.shape[:2]
    for hand in result.hand_landmarks:
        # draw connections
        for a, b in CONNECTIONS:
            x1, y1 = int(hand[a].x * w), int(hand[a].y * h)
            x2, y2 = int(hand[b].x * w), int(hand[b].y * h)
            cv2.line(frame, (x1, y1), (x2, y2), (180, 180, 180), 2)
        # draw each landmark dot
        for i, lm in enumerate(hand):
            cx, cy = int(lm.x * w), int(lm.y * h)
            color = (0, 255, 0) if i == 8 else (0, 255, 255)  # green for index tip
            cv2.circle(frame, (cx, cy), 5 if i != 8 else 10, color, -1)
    return frame

# ── setup landmarker ───────────────────────────────────────────────────────────
options = vision.HandLandmarkerOptions(
    base_options=python.BaseOptions(model_asset_path="hand_landmarker.task"),
    running_mode=vision.RunningMode.VIDEO,
    num_hands=1,
    min_hand_detection_confidence=0.5,
    min_hand_presence_confidence=0.5,
    min_tracking_confidence=0.5
)

# ── webcam loop ────────────────────────────────────────────────────────────────
cap = cv2.VideoCapture(0)
timestamp_ms = 0

with vision.HandLandmarker.create_from_options(options) as landmarker:
    while cap.isOpened():
        success, frame = cap.read()
        if not success:
            continue

        timestamp_ms += 33

        rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb)
        result = landmarker.detect_for_video(mp_image, timestamp_ms)

        if result.hand_landmarks:
            h, w = frame.shape[:2]
            tip = result.hand_landmarks[0][8]
            cx, cy = int(tip.x * w), int(tip.y * h)
            print(f"Index tip: ({cx}, {cy})")
            frame = draw_hands(frame, result)

        cv2.imshow("Hand Tracking", cv2.flip(frame, 1))
        if cv2.waitKey(5) & 0xFF == 27:
            break

cap.release()
cv2.destroyAllWindows()

Index tip: (223, 395)
Index tip: (218, 395)
Index tip: (219, 393)
Index tip: (216, 394)
Index tip: (215, 395)
Index tip: (213, 396)
Index tip: (212, 395)
Index tip: (212, 396)
Index tip: (210, 397)
Index tip: (211, 399)
Index tip: (211, 398)
Index tip: (212, 399)
Index tip: (210, 398)
Index tip: (210, 398)
Index tip: (211, 399)
Index tip: (208, 400)
Index tip: (206, 393)
Index tip: (229, 229)
Index tip: (233, 233)
Index tip: (219, 229)
Index tip: (221, 231)
Index tip: (221, 232)
Index tip: (221, 230)
Index tip: (222, 229)
Index tip: (226, 229)
Index tip: (226, 229)
Index tip: (229, 230)
Index tip: (229, 229)
Index tip: (230, 229)
Index tip: (229, 229)
Index tip: (228, 229)
Index tip: (229, 229)
Index tip: (230, 230)
Index tip: (230, 230)
Index tip: (230, 230)
Index tip: (229, 229)
Index tip: (229, 229)
Index tip: (229, 229)
Index tip: (229, 230)
Index tip: (229, 230)
Index tip: (228, 229)
Index tip: (228, 229)
Index tip: (228, 230)
Index tip: (228, 230)
Index tip: (228, 230)
Index tip:

: 

In [ ]:
import cv2
import mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python import vision
import numpy as np
import math
from collections import deque

# ── download model ─────────────────────────────────────────────────────────────
import urllib.request, os
if not os.path.exists("hand_landmarker.task"):
    print("Downloading model...")
    urllib.request.urlretrieve(
        "https://storage.googleapis.com/mediapipe-models/hand_landmarker/hand_landmarker/float16/1/hand_landmarker.task",
        "hand_landmarker.task"
    )

# ── smoothing buffer ───────────────────────────────────────────────────────────
# Instead of using raw x,y every frame, we keep the last N positions
# and average them. This kills the shakiness.
SMOOTH_N = 5          # how many frames to average — increase if still shaky
x_buffer = deque(maxlen=SMOOTH_N)
y_buffer = deque(maxlen=SMOOTH_N)

def smoothed_point(raw_x, raw_y):
    """Add new point to buffer, return the rolling average."""
    x_buffer.append(raw_x)
    y_buffer.append(raw_y)
    return int(np.mean(x_buffer)), int(np.mean(y_buffer))

# ── pinch detection with hysteresis ───────────────────────────────────────────
# Instead of one threshold (causes flickering at the boundary),
# use TWO thresholds: one to go UP, one to go DOWN.
# Pen only lifts when dist < PEN_UP, only draws when dist > PEN_DOWN.
# The gap between them creates a "dead zone" that stops flickering.
PEN_UP_DIST   = 35    # pinch tighter than this = pen lifts
PEN_DOWN_DIST = 60    # spread wider than this = pen draws
pen_down = False      # current pen state

def get_distance(lm1, lm2, w, h):
    x1, y1 = lm1.x * w, lm1.y * h
    x2, y2 = lm2.x * w, lm2.y * h
    return math.sqrt((x2-x1)**2 + (y2-y1)**2)

# ── setup ──────────────────────────────────────────────────────────────────────
options = vision.HandLandmarkerOptions(
    base_options=python.BaseOptions(model_asset_path="hand_landmarker.task"),
    running_mode=vision.RunningMode.VIDEO,
    num_hands=1,
    min_hand_detection_confidence=0.5,
    min_hand_presence_confidence=0.5,
    min_tracking_confidence=0.5
)

cap = cv2.VideoCapture(0)
ret, test_frame = cap.read()
h, w = test_frame.shape[:2]

canvas = np.zeros((h, w, 3), dtype=np.uint8)
prev_x, prev_y = None, None
timestamp_ms = 0

# how many frames the hand can disappear before we break the stroke
# fixes the "broken line when detection drops for 1 frame" problem
MAX_MISSING_FRAMES = 5
missing_frames = 0

DRAW_COLOR  = (255, 255, 255)
BRUSH_SIZE  = 6

with vision.HandLandmarker.create_from_options(options) as landmarker:
    while cap.isOpened():
        success, frame = cap.read()
        if not success:
            continue

        timestamp_ms += 33
        frame = cv2.flip(frame, 1)

        rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb)
        result = landmarker.detect_for_video(mp_image, timestamp_ms)

        if result.hand_landmarks:
            missing_frames = 0   # reset the missing counter — hand is visible
            hand = result.hand_landmarks[0]

            index_tip = hand[8]
            thumb_tip = hand[4]

            # raw pixel position of index tip
            raw_x = int(index_tip.x * w)
            raw_y = int(index_tip.y * h)

            # smoothed position — this is what we actually use
            cx, cy = smoothed_point(raw_x, raw_y)

            # pinch distance
            dist = get_distance(thumb_tip, index_tip, w, h)

            # ── hysteresis pen state ───────────────────────────────────────────
            

            if pen_down and dist < PEN_UP_DIST:
                pen_down = False     # was drawing, now pinching — lift pen
            elif not pen_down and dist > PEN_DOWN_DIST:
                pen_down = True      # was lifted, now spread — put pen down

            if not pen_down:
                # pen up — show red dot, reset prev so no ghost line on resume
                cv2.circle(frame, (cx, cy), BRUSH_SIZE, (0, 255, 0), -1)

                if prev_x is not None:
                    # skip drawing if jump is too large (detection glitch)
                    # this prevents sudden long lines when tracking hiccups
                    jump = math.sqrt((cx - prev_x)**2 + (cy - prev_y)**2)
                    if jump < 80:   # ignore if fingertip jumped more than 80px
                        cv2.line(canvas, (prev_x, prev_y), (cx, cy),
                                 DRAW_COLOR, BRUSH_SIZE * 2)

                prev_x, prev_y = cx, cy
            else:
                # pen down — draw
                
                prev_x, prev_y = None, None
                cv2.circle(frame, (cx, cy), 15, (0, 0, 255), -1)
                cv2.putText(frame, "PEN UP", (cx+20, cy),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 0, 255), 2)

        else:
            # hand not detected this frame
            missing_frames += 1
            if missing_frames > MAX_MISSING_FRAMES:
                # hand has been gone long enough — break the stroke
                prev_x, prev_y = None, None
                x_buffer.clear()
                y_buffer.clear()

        # ── overlay canvas on camera ───────────────────────────────────────────
        gray = cv2.cvtColor(canvas, cv2.COLOR_BGR2GRAY)
        _, mask = cv2.threshold(gray, 10, 255, cv2.THRESH_BINARY)
        frame[mask == 255] = canvas[mask == 255]

        # ── UI text ────────────────────────────────────────────────────────────
        state_text = "DRAWING" if pen_down else "PEN UP (pinch to lift)"
        state_color = (0, 255, 0) if pen_down else (0, 0, 255)
        cv2.putText(frame, state_text, (10, 35),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.8, state_color, 2)
        cv2.putText(frame, "C = clear  |  ESC = quit", (10, h - 15),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.55, (150, 150, 150), 1)

        cv2.imshow("AirCanvas", frame)

        key = cv2.waitKey(1) & 0xFF   # waitKey(1) not (5) — faster loop
        if key == 27:
            break
        elif key == ord('c'):
            canvas[:] = 0
            prev_x, prev_y = None, None
            x_buffer.clear()
            y_buffer.clear()

cap.release()
cv2.destroyAllWindows()